# FD003 - LightGBM Hyperparameter Refinement

**Advertencia metodologica:** este notebook usa solo `train_FD003.txt` con validacion artificial por motores completos y cortes internos. El test oficial no se utiliza.

Objetivo: probar si el candidato final interno actual de FD003 mejora ajustando solo hiperparametros internos de LightGBM, manteniendo fijo `feature_set=fault_sensitive`, `window_size=50`, `rul_cap=125`, `objective=quantile`, `alpha=0.4` y `sample_weight_scheme=none`.

## 1. Setup

Se cargan utilidades del proyecto, se detecta la raiz y se crean las carpetas de salida obligatorias.

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "CMAPSSData").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.fd003_improvement_utils import (
    FD003_RANDOM_STATES,
    comparison_row_from_summary,
    config_from_summary_row,
    ensure_fd003_dirs,
    evaluate_fd003_config_split,
    fd003_lgbm_hyperparam_refinement_configs,
    is_defensible_fd003_refinement,
    load_json,
    save_json,
    select_best_summary_row,
    summarize_detail,
)

PATHS = ensure_fd003_dirs(PROJECT_ROOT)
DATA_DIR = PROJECT_ROOT / "CMAPSSData"
RESULTS_DIR = PATHS["results"]
FIGURES_DIR = PATHS["figures"]
CONFIGS_DIR = PATHS["configs"]
NOTES_DIR = PATHS["notes"]

print(f"Proyecto: {PROJECT_ROOT}")
print(f"Resultados: {RESULTS_DIR}")
print(f"Random states: {FD003_RANDOM_STATES}")

## 2. Configuracion base fija

Se usa el candidato final actual de FD003. Si no existe, se usa como fallback la config de features sensibles. El experimento solo modifica hiperparametros internos de LightGBM.

In [ ]:
final_config_path = CONFIGS_DIR / "fd003_final_candidate_config.json"
fallback_config_path = CONFIGS_DIR / "fd003_fault_sensitive_features_best_config.json"

if final_config_path.exists():
    base_config = load_json(final_config_path)
    base_config_source = final_config_path.name
elif fallback_config_path.exists():
    base_config = load_json(fallback_config_path)
    base_config_source = fallback_config_path.name
else:
    raise FileNotFoundError("No existe fd003_final_candidate_config.json ni fd003_fault_sensitive_features_best_config.json")

required_fixed = {
    "window_size": 50,
    "rul_cap": 125,
    "objective": "quantile",
    "alpha": 0.4,
    "sample_weight_scheme": "none",
    "feature_set": "fault_sensitive",
}
for key, expected in required_fixed.items():
    actual = base_config.get(key)
    if actual != expected:
        raise ValueError(f"Config base no respeta {key}={expected}. Valor encontrado: {actual}")

print("Config base:", base_config_source)
print(json.dumps({
    "model_name": base_config["model_name"],
    "window_size": base_config["window_size"],
    "rul_cap": base_config["rul_cap"],
    "objective": base_config["objective"],
    "alpha": base_config["alpha"],
    "sample_weight_scheme": base_config["sample_weight_scheme"],
    "feature_set": base_config["feature_set"],
    "hyperparameters": base_config["hyperparameters"],
}, indent=2))

## 3. Busqueda acotada

Se hace randomized search reproducible con 24 combinaciones como maximo, incluyendo siempre la configuracion actual. No se prueban clusters, probabilidades de cluster, mixture of experts ni nuevas features.

In [ ]:
configs = fd003_lgbm_hyperparam_refinement_configs(
    base_config=base_config,
    n_configs=24,
    random_state=2026,
)

print(f"Cantidad de combinaciones: {len(configs)}")
assert len(configs) <= 30
print("Primera combinacion incluida:", configs[0]["hyperparameter_search_label"])
print(json.dumps(configs[0]["hyperparameters"], indent=2))

## 4. Ejecucion multi-split

Cada combinacion se evalua con los mismos 5 random states y el mismo protocolo de validacion artificial por motores completos.

In [ ]:
detail_rows = []
for config_index, config in enumerate(configs, start=1):
    print(f"Evaluando {config_index}/{len(configs)}: {config['model_name']}")
    for state in FD003_RANDOM_STATES:
        metrics, _, prepared = evaluate_fd003_config_split(
            config=config,
            data_dir=DATA_DIR,
            random_state=state,
        )
        metrics["config_index"] = config_index - 1
        metrics["hyperparameter_search_label"] = config["hyperparameter_search_label"]
        for key, value in config["hyperparameters"].items():
            metrics[key] = value
        detail_rows.append(metrics)

detail = pd.DataFrame(detail_rows)

group_cols = [
    "config_index",
    "hyperparameter_search_label",
    "model_name",
    "approach",
    "window_size",
    "rul_cap",
    "objective",
    "alpha",
    "sample_weight_scheme",
    "feature_set",
    "num_leaves",
    "min_child_samples",
    "reg_lambda",
    "reg_alpha",
    "subsample",
    "colsample_bytree",
    "learning_rate",
    "n_estimators",
    "max_depth",
]
summary = summarize_detail(detail, group_cols).sort_values(
    ["mean_cmapss_score", "worst_cmapss_score", "mean_dangerous_error_pct", "mean_rmse"]
).reset_index(drop=True)

detail.to_csv(RESULTS_DIR / "fd003_lgbm_hyperparam_refinement_detail.csv", index=False)
summary.to_csv(RESULTS_DIR / "fd003_lgbm_hyperparam_refinement_summary.csv", index=False)

print("Top modelos por C-MAPSS:")
display(summary.head(10))

## 5. Seleccion prudente

Se compara el mejor refinamiento contra el candidato actual. Solo se acepta reemplazo si mejora C-MAPSS de forma defendible, no empeora dangerous error de forma clara y no empeora worst C-MAPSS.

In [ ]:
current_row = summary.loc[summary["hyperparameter_search_label"] == "baseline_current"].iloc[0].to_dict()
best_row = select_best_summary_row(summary)
clear_improvement = is_defensible_fd003_refinement(best_row, current_row)
selected_row = best_row if clear_improvement else current_row

comparison_vs_current = pd.DataFrame([
    {"row_type": "current_candidate", **current_row},
    {"row_type": "best_refinement", **best_row},
])
for metric in ["mean_cmapss_score", "worst_cmapss_score", "mean_dangerous_error_pct", "mean_rmse", "mean_mae", "mean_bias_mean"]:
    comparison_vs_current.loc[comparison_vs_current["row_type"] == "best_refinement", f"delta_vs_current_{metric}"] = best_row[metric] - current_row[metric]
comparison_vs_current.to_csv(RESULTS_DIR / "fd003_lgbm_hyperparam_refinement_comparison_vs_current.csv", index=False)

best_config = config_from_summary_row(best_row, hyperparameters={
    "colsample_bytree": best_row["colsample_bytree"],
    "learning_rate": best_row["learning_rate"],
    "max_depth": int(best_row["max_depth"]),
    "min_child_samples": int(best_row["min_child_samples"]),
    "n_estimators": int(best_row["n_estimators"]),
    "num_leaves": int(best_row["num_leaves"]),
    "reg_alpha": best_row["reg_alpha"],
    "reg_lambda": best_row["reg_lambda"],
    "subsample": best_row["subsample"],
})
best_config.update({
    "selected_from": "notebooks/FD003/modeling/24_fd003_lgbm_hyperparam_refinement.ipynb",
    "base_config_source": base_config_source,
    "validation_protocol": "train_FD003 only, held-out units, artificial cutoffs, multi-split",
    "official_test_used": False,
    "official_test_used_for_selection": False,
    "clear_improvement_over_current": bool(clear_improvement),
    "selection_note": "Refinement replaces current candidate only if C-MAPSS and robustness improve defensibly.",
    "decision": "replace_current_candidate" if clear_improvement else "keep_current_candidate",
    "validation_summary": {k: (None if pd.isna(v) else v) for k, v in best_row.items()},
})
if not clear_improvement:
    best_config["note"] = "No se reemplaza el candidato final actual: el mejor refinamiento no mejora de forma defendible frente al modelo fault-sensitive actual."

save_json(CONFIGS_DIR / "fd003_lgbm_hyperparam_refinement_best_config.json", best_config)

if clear_improvement:
    refined_final = dict(best_config)
    refined_final["selected_as"] = "FD003 final internal candidate refined"
    save_json(CONFIGS_DIR / "fd003_final_candidate_config_refined.json", refined_final)

print("Mejora clara:", clear_improvement)
print("Actual:")
display(pd.DataFrame([current_row]))
print("Mejor refinamiento:")
display(pd.DataFrame([best_row]))

## 6. Figuras

Se guardan visuales de comparacion entre combinaciones, priorizando C-MAPSS y robustez.

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")

top = summary.head(10).copy()
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top["model_name"], top["mean_cmapss_score"], xerr=top["std_cmapss_score"], color="#4c78a8")
ax.invert_yaxis()
ax.set_xlabel("Mean C-MAPSS score")
ax.set_title("FD003 LGBM refinement - top models by C-MAPSS")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fd003_lgbm_refinement_top_models_cmapss.png", dpi=160)
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(summary["mean_rmse"], summary["mean_cmapss_score"], s=50, alpha=0.8)
ax.scatter(current_row["mean_rmse"], current_row["mean_cmapss_score"], s=120, marker="*", color="red", label="current")
ax.set_xlabel("Mean RMSE")
ax.set_ylabel("Mean C-MAPSS")
ax.set_title("FD003 LGBM refinement - RMSE vs C-MAPSS")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fd003_lgbm_refinement_rmse_vs_cmapss.png", dpi=160)
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(summary["mean_dangerous_error_pct"], summary["mean_cmapss_score"], s=50, alpha=0.8)
ax.scatter(current_row["mean_dangerous_error_pct"], current_row["mean_cmapss_score"], s=120, marker="*", color="red", label="current")
ax.set_xlabel("Mean dangerous error (%)")
ax.set_ylabel("Mean C-MAPSS")
ax.set_title("FD003 LGBM refinement - dangerous vs C-MAPSS")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fd003_lgbm_refinement_dangerous_vs_cmapss.png", dpi=160)
plt.show()

compare_plot = pd.DataFrame([
    {"label": "current", "worst_cmapss_score": current_row["worst_cmapss_score"]},
    {"label": "best_refinement", "worst_cmapss_score": best_row["worst_cmapss_score"]},
    {"label": "selected", "worst_cmapss_score": selected_row["worst_cmapss_score"]},
])
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(compare_plot["label"], compare_plot["worst_cmapss_score"], color=["#e45756", "#4c78a8", "#54a24b"])
ax.set_ylabel("Worst C-MAPSS across splits")
ax.set_title("Worst C-MAPSS comparison")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fd003_lgbm_refinement_worst_cmapss_comparison.png", dpi=160)
plt.show()

## 7. Interpretacion y tabla global refinada

Se guarda una interpretacion breve y una comparacion global que incorpora este ultimo refinamiento.

In [ ]:
def comparison_row(row, approach, notes):
    return {
        "model_name": row.get("model_name"),
        "approach": approach,
        "window_size": row.get("window_size"),
        "rul_cap": row.get("rul_cap"),
        "objective": row.get("objective"),
        "alpha": row.get("alpha"),
        "sample_weight_scheme": row.get("sample_weight_scheme"),
        "feature_set": row.get("feature_set"),
        "mean_MAE": row.get("mean_mae"),
        "std_MAE": row.get("std_mae"),
        "mean_RMSE": row.get("mean_rmse"),
        "std_RMSE": row.get("std_rmse"),
        "mean_R2": row.get("mean_r2"),
        "std_R2": row.get("std_r2"),
        "mean_CMAPSS": row.get("mean_cmapss_score"),
        "std_CMAPSS": row.get("std_cmapss_score"),
        "mean_dangerous_error_pct": row.get("mean_dangerous_error_pct"),
        "mean_conservative_error_pct": row.get("mean_conservative_error_pct"),
        "mean_bias": row.get("mean_bias_mean"),
        "worst_RMSE": row.get("worst_rmse"),
        "worst_CMAPSS": row.get("worst_cmapss_score"),
        "notes": notes,
    }

transfer_summary = pd.read_csv(RESULTS_DIR / "fd003_transfer_fd001_pipeline_validation_summary.csv")
short_summary = pd.read_csv(RESULTS_DIR / "fd003_short_tuning_summary.csv")
fault_summary = pd.read_csv(RESULTS_DIR / "fd003_fault_sensitive_features_summary.csv")
pseudo_summary = pd.read_csv(RESULTS_DIR / "fd003_pseudo_cluster_experiments_summary.csv")

transfer_row = transfer_summary.iloc[0].to_dict()
transfer_row["feature_set"] = "base"
short_row = short_summary.sort_values(["mean_cmapss_score", "mean_dangerous_error_pct", "mean_rmse"]).iloc[0].to_dict()
fault_row = fault_summary.sort_values(["mean_cmapss_score", "mean_dangerous_error_pct", "mean_rmse"]).iloc[0].to_dict()
pseudo_row = pseudo_summary.sort_values(["mean_cmapss_score", "mean_dangerous_error_pct", "mean_rmse"]).iloc[0].to_dict()
refinement_row = best_row
final_selected_row = selected_row

rows = [
    comparison_row(transfer_row, "FD001 transfer baseline sobre FD003", "Pipeline FD001 transferido directo a FD003."),
    comparison_row(short_row, "FD003 short tuning best", "Mejor punto de tuning corto FD003."),
    comparison_row(fault_row, "FD003 fault-sensitive features best", "Candidato actual antes del refinamiento."),
    comparison_row(pseudo_row, "FD003 pseudo-cluster best", "Mejor variante pseudo-cluster; no necesariamente seleccionada."),
    comparison_row(current_row, "FD003 current final candidate", "Candidato final actual usado como baseline del refinamiento."),
    comparison_row(refinement_row, "FD003 hyperparam refinement best", "Mejor combinacion de hiperparametros LightGBM de esta busqueda acotada."),
    comparison_row(final_selected_row, "FD003 final selected candidate", "Seleccion final tras refinamiento; mantiene el actual si no hay mejora defendible."),
]
global_refined = pd.DataFrame(rows)
global_refined.to_csv(RESULTS_DIR / "fd003_model_improvement_global_comparison_refined.csv", index=False)

cmapss_delta = current_row["mean_cmapss_score"] - best_row["mean_cmapss_score"]
worst_delta = current_row["worst_cmapss_score"] - best_row["worst_cmapss_score"]
dangerous_delta = best_row["mean_dangerous_error_pct"] - current_row["mean_dangerous_error_pct"]
rmse_delta = best_row["mean_rmse"] - current_row["mean_rmse"]

lines = [
    "FD003 - Refinamiento acotado de hiperparametros LightGBM",
    "",
    "Se mantuvo fijo el enfoque del candidato actual: LightGBM, feature_set fault_sensitive, window_size=50, rul_cap=125, objective quantile, alpha=0.4 y sin sample weights.",
    "No se usaron clusters, cluster probabilities, mixture of experts ni test oficial.",
    "La validacion fue la misma de los notebooks anteriores: train_FD003, motores completos held-out, cortes artificiales y random states [0, 1, 2, 3, 4].",
    "",
    f"Se evaluaron {len(configs)} combinaciones de hiperparametros internos de LightGBM, incluyendo siempre la configuracion actual.",
    "Los hiperparametros refinados fueron num_leaves, min_child_samples, reg_lambda, reg_alpha, subsample y colsample_bytree. Se mantuvieron learning_rate=0.03, n_estimators=1300 y max_depth=-1.",
    "",
    "Candidato actual:",
    f"- C-MAPSS {current_row['mean_cmapss_score']:.3f}, worst C-MAPSS {current_row['worst_cmapss_score']:.3f}, RMSE {current_row['mean_rmse']:.3f}, dangerous {current_row['mean_dangerous_error_pct']:.2f}%.",
    "Mejor refinamiento encontrado:",
    f"- {best_row['model_name']}: C-MAPSS {best_row['mean_cmapss_score']:.3f}, worst C-MAPSS {best_row['worst_cmapss_score']:.3f}, RMSE {best_row['mean_rmse']:.3f}, dangerous {best_row['mean_dangerous_error_pct']:.2f}%.",
    "",
    f"Deltas best - current: C-MAPSS {-cmapss_delta:.3f}, worst C-MAPSS {-worst_delta:.3f}, RMSE {rmse_delta:.3f}, dangerous {dangerous_delta:.2f} puntos porcentuales.",
    "",
]

if clear_improvement:
    lines.append("Decision: el refinamiento mejora de forma defendible y se guarda como candidato final refinado.")
else:
    lines.append("Decision: no se reemplaza el candidato final actual. El mejor refinamiento no mejora de forma suficientemente defendible considerando C-MAPSS, worst C-MAPSS, dangerous error y estabilidad.")

lines.extend([
    "",
    "Conclusion prudente: este notebook sirve como cierre de robustez. Si no hay mejora clara, se mantiene el modelo fault-sensitive actual por simplicidad y estabilidad.",
    "",
])
(NOTES_DIR / "fd003_lgbm_hyperparam_refinement_interpretation.txt").write_text("
".join(lines), encoding="utf-8")

print("
".join(lines))
display(global_refined)